In [0]:
CREATE SCHEMA IF NOT EXISTS gold_olist;


In [0]:
USE SCHEMA silver_olist

In [0]:
SHOW TABLES

#####1.Modify & load dimension tables to gold_layer: customers, products, sellers,payment,reviews

1.1. dim_customer

In [0]:
select * from orders limit 1

In [0]:
CREATE TABLE IF NOT EXISTS gold_olist.dim_customer AS
WITH ranked_customers AS(
  SELECT
    customer_unique_id,
    zip_code_prefix,
    city,
    state,
    ROW_NUMBER() OVER (
        PARTITION BY customer_unique_id
        ORDER BY order_purchase_ts DESC) AS row_number--Count row_number to choose the latest info row of customer
  FROM customers c
  JOIN orders o
      ON c.customer_id = o.customer_id)
--Select the info of the customer having row_number=1
SELECT
    customer_unique_id,
    zip_code_prefix,
    city,
    state
FROM ranked_customers
WHERE row_number = 1;

1.2. dim_seller

In [0]:
-- Count row for each seller_id; then, only choose 1 record for each seller_id
CREATE TABLE IF NOT EXISTS gold_olist.dim_seller AS
  WITH rank_seller AS
    (SELECT
          seller_id,
          zip_code_prefix,
          city,
          state,
          latitude,
          longitude,
          ROW_NUMBER() OVER (
              PARTITION BY seller_id ORDER BY seller_id)
          AS rn,
          COUNT(*) OVER (PARTITION BY seller_id) AS cnt
      FROM sellers
      order by seller_id,rn)
    SELECT 
      seller_id,
      zip_code_prefix,
      city,
      state,
      latitude,
      longitude
    FROM rank_seller 
    WHERE rn=cnt --Only choose the latest record in the table for each seller_id


1.3 dim_product

In [0]:
SELECT * FROM products LIMIT 5

In [0]:
CREATE TABLE IF NOT EXISTS gold_olist.dim_product AS
SELECT *
FROM products

1.4. Dim_payment

In [0]:
CREATE TABLE IF NOT EXISTS gold_olist.dim_payment AS
  SELECT * FROM order_payments

1.5. Dim_review

In [0]:
CREATE TABLE IF NOT EXISTS gold_olist.dim_review AS
SELECT * 
FROM order_reviews

#####2. Fact_orders & Fact_order_item

2.1. Fact_order_item

In [0]:
--Load order_items to gold layer
CREATE TABLE IF NOT EXISTS gold_olist.fact_order_item AS 
SELECT * FROM order_items

2.1. Fact_orders

In [0]:
--Calculate each order value
SELECT * FROM order_items limit 5

In [0]:
CREATE TABLE IF NOT EXISTS gold_olist.fact_order AS
WITH order_details AS --calculate details of each order such as total_items, goods_values, shipping_fee, and payment_amount
  (SELECT order_id, 
  COUNT(product_id) as total_items,
  ROUND(SUM(price),2) as goods_value, 
  ROUND(SUM(freight_value),2) as shipping_fee,
  ROUND(SUM(freight_value+price),2) as payment_amount
  FROM order_items
  GROUP BY order_id),
payment_details AS(--calculate paid value of each order 
  SELECT order_id,
  ROUND(SUM(payment_value),2) as paid
  FROM order_payments
  GROUP BY order_id
),
new_order_id AS--Left join orders with order_details and payment_details ctes. 
  (SELECT 
      o.order_id,
      o.order_status,
      o.order_purchase_ts,
      o.order_purchase_date,
      o.order_approved_ts,
      o.order_approved_date,
      o.order_delivered_carrier_ts,
      o.order_delivered_carrier_date,
      o.order_delivered_customer_ts,
      o.order_delivered_customer_date,
      o.order_estimated_delivery_date,
      COALESCE(od.total_items,0) AS total_items,
      COALESCE(od.goods_value,0) AS goods_values,
      COALESCE(od.shipping_fee,0) AS shipping_fee,
      COALESCE(od.payment_amount,0) AS payment_amount,
      COALESCE(pd.paid,0) AS paid
  FROM orders o
  LEFT JOIN order_details od
  ON o.order_id = od.order_id
  LEFT JOIN payment_details pd
  ON o.order_id = pd.order_id
  )
--Remove invaid rows that having paid amount more than payment_amount of each order, it is possible that customer has paid 1 cent more than payment_amount. 
  SELECT * FROM new_order_id 
  WHERE paid<= payment_amount+0.01 

#####3. Download all tables in gold layers to csv